In [ ]:
## This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Cài đặt công cụ giải nén 7z (trong trường hợp máy chủ Kaggle chưa cài sẵn)
!apt-get install p7zip-full -y

# Giải nén train.7z và test.7z vào thư mục làm việc hiện tại (/kaggle/working/)
!7za x /kaggle/input/competitions/cifar-10/train.7z -o/kaggle/working/ -y

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Đọc file CSV chứa nhãn
df = pd.read_csv('/kaggle/input/competitions/cifar-10/trainLabels.csv')

# Chuyển đổi cột 'id' từ dạng số nguyên (ví dụ: 1) sang định dạng tên file ảnh (ví dụ: '1.png')
df['id'] = df['id'].astype(str) + '.png'

# 2. Thiết lập ImageDataGenerator
# Lệnh này thực hiện 2 việc: Chuẩn hóa pixel về khoảng [0, 1] và tự động trích ra 20% dữ liệu để làm tập kiểm thử (validation)
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Đường dẫn tới thư mục ảnh huấn luyện vừa được giải nén
train_dir = '/kaggle/working/train/'

# 3. Tạo luồng dữ liệu cho tập Train (Huấn luyện)
train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory=train_dir,
    x_col='id',         # Cột chứa tên tệp ảnh
    y_col='label',      # Cột chứa nhãn phân loại
    subset='training',  # Lấy 80% dữ liệu
    batch_size=32,
    seed=42,
    shuffle=True,
    class_mode='categorical', # Mã hóa nhãn dạng chữ sang dạng one-hot
    target_size=(32, 32)
)

# 4. Tạo luồng dữ liệu cho tập Validation (Kiểm định)
val_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory=train_dir,
    x_col='id',
    y_col='label',
    subset='validation', # Lấy 20% dữ liệu còn lại
    batch_size=32,
    seed=42,
    shuffle=True,
    class_mode='categorical',
    target_size=(32, 32)
)

# 1. Khởi tạo lại luồng dữ liệu kiểm định KHÔNG xáo trộn (shuffle=False)
val_generator_eval = datagen.flow_from_dataframe(
    dataframe=df,
    directory=train_dir,
    x_col='id',
    y_col='label',
    subset='validation',
    batch_size=32,
    seed=42,
    shuffle=False,       # <--- Không xáo trộn để đối chiếu chính xác
    class_mode='categorical',
    target_size=(32, 32)
)

In [ ]:
from tensorflow.keras import models, layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    # --- Khối Tích chập 1 (Học các chi tiết cơ bản) ---
    # Thêm padding='same' để không bị mất viền của bức ảnh
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(), # Chuẩn hóa dữ liệu
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25), # Ngẫu nhiên "tắt" 25% nơ-ron để chống học vẹt

    # --- Khối Tích chập 2 (Học các đặc trưng phức tạp hơn) ---
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # --- Khối Tích chập 3 (Nâng cao sức mạnh - Tùy chọn nhưng khuyên dùng) ---
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # --- Khối Phân loại (Đưa ra kết luận) ---
    layers.Flatten(),
    layers.Dense(128, activation='relu'), # Tăng lên 128 nơ-ron vì khối tích chập đã phức tạp hơn
    layers.BatchNormalization(),
    layers.Dropout(0.5), # Ở lớp Fully Connected, Dropout mạnh tay hơn (50%)
    layers.Dense(10, activation='softmax')
])

# Biên dịch mô hình
model.compile(optimizer='adam',
              loss='categorical_crossentropy', # <-- Đổi từ SparseCategoricalCrossentropy sang CategoricalCrossentropy
              metrics=['accuracy'])

# 1. Cài đặt Early Stopping (Dừng sớm nếu không cải thiện)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# 2. Cài đặt Checkpoint (Chỉ lưu model có điểm Validation Accuracy cao nhất)
checkpoint = ModelCheckpoint(
    filepath='/kaggle/working/best_cifar10_model.keras', # Tên file sẽ lưu
    monitor='val_accuracy',
    save_best_only=True,    # Chỉ lưu nếu kết quả tốt hơn lần trước
    mode='max',
    verbose=1               # In ra thông báo mỗi lần lưu
)

# 3. Gắn cả 2 công cụ này vào hàm fit
history = model.fit(
    train_generator,
    validation_data=val_generator_eval, # Đảm bảo generator validation không shuffle
    epochs=100,
    callbacks=[early_stop, checkpoint]  # <--- Khai báo tại đây
)

In [ ]:
import matplotlib.pyplot as plt

# Trích xuất dữ liệu từ lịch sử huấn luyện
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

# Vẽ biểu đồ Độ chính xác (Accuracy)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, 'b-', label='Độ chính xác (Train)')
plt.plot(epochs, val_acc, 'r-', label='Độ chính xác (Validation)')
plt.title('Biểu đồ Độ chính xác')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Vẽ biểu đồ Độ mất mát (Loss)
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'b-', label='Sai số (Train)')
plt.plot(epochs, val_loss, 'r-', label='Sai số (Validation)')
plt.title('Biểu đồ Sai số (Loss)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# --- CELL MỚI: DÀNH RIÊNG CHO ĐÁNH GIÁ (EVALUATION) ---

# 2. Lấy nhãn thực tế và dự đoán
print("\nĐang tiến hành dự đoán trên tập Validation...")
y_true = val_generator_eval.classes
predictions = model.predict(val_generator_eval)
y_pred = np.argmax(predictions, axis=1)
class_names = list(val_generator_eval.class_indices.keys())

# 3. In Báo cáo phân loại (Classification Report)
print("\n" + "="*50)
print("BÁO CÁO PHÂN LOẠI CHI TIẾT (CLASSIFICATION REPORT)")
print("="*50)
print(classification_report(y_true, y_pred, target_names=class_names))

# 4. Vẽ Ma trận nhầm lẫn (Confusion Matrix)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))

# Cấu hình biểu đồ heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)

plt.title('Ma trận Nhầm lẫn (Confusion Matrix)', fontsize=14)
plt.ylabel('Nhãn thực tế (True Label)', fontsize=12)
plt.xlabel('Nhãn dự đoán (Predicted Label)', fontsize=12)
plt.xticks(rotation=45) # Xoay nhãn trục X một chút để chữ không bị đè lên nhau
plt.tight_layout()
plt.show()